# W6: 재고 예측 & 발주량 자동 계산

이 노트북은 F&B 매장의 재고 예측 및 발주량을 자동으로 계산하는 도구입니다.

## Step 0: 라이브러리 설치 및 폰트 설정

In [ ]:
# 필요한 라이브러리 설치
!pip install pandas matplotlib numpy -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
from datetime import datetime, timedelta
import io
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
def setup_korean_font():
    try:
        # 시스템에 설치된 폰트 찾기
        font_files = fm.findSystemFonts()
        korean_fonts = []
        
        # 한국어 폰트 찾기
        for font_file in font_files:
            try:
                font_prop = fm.FontProperties(fname=font_file)
                font_name = font_prop.get_name()
                # 한국어 폰트인지 확인 (일반적인 한국어 폰트 이름 포함)
                if any(keyword in font_name.lower() for keyword in ['malgun', 'gulim', 'dotum', 'batang', 'nanum', 'apple sd gothic', 'noto sans']):
                    korean_fonts.append(font_file)
                    print(f"한국어 폰트 찾음: {font_name} - {font_file}")
            except:
                continue
        
        if korean_fonts:
            # 찾은 첫 번째 한국어 폰트 사용
            selected_font = korean_fonts[0]
            fm.fontManager.addfont(selected_font)
            plt.rc('font', family=fm.FontProperties(fname=selected_font).get_name())
            print(f"폰트 설정 완료: {fm.FontProperties(fname=selected_font).get_name()}")
        else:
            # 한국어 폰트가 없는 경우 기본 폰트 사용
            print("경고: 시스템에서 한국어 폰트를 찾지 못했습니다. 영문으로 표시됩니다.")
            plt.rc('font', family='DejaVu Sans')
    except Exception as e:
        print(f"폰트 설정 중 오류 발생: {e}")
        plt.rc('font', family='DejaVu Sans')

# 마이너스 부호 표시 설정
plt.rcParams['axes.unicode_minus'] = False

# 폰트 설정 실행
setup_korean_font()

## Step 1: 재료 정보 입력

In [ ]:
# 재료 정보 입력 함수
def input_ingredients():
    ingredients = []
    print("📦 재료 정보를 입력하세요 (입력 완료시 '완료' 입력)")
    
    while True:
        print(f"\n--- 재료 {len(ingredients)+1} ---")
        name = input("재료 이름: ").strip()
        
        if name.lower() == '완료':
            break
            
        if not name:
            print("재료 이름을 입력해주세요.")
            continue
            
        try:
            current_stock = float(input(f"{name} 현재 재고량: "))
            daily_usage = float(input(f"{name} 일일 사용량: "))
            lead_time = int(input(f"{name} 발주 소요일수: "))
            
            if current_stock <= 0 or daily_usage <= 0 or lead_time <= 0:
                print("모든 값은 0보다 커야 합니다.")
                continue
                
            ingredients.append({
                'name': name,
                'current_stock': current_stock,
                'daily_usage': daily_usage,
                'lead_time_days': lead_time
            })
            
            print(f"✅ {name} 재료 정보 추가 완료")
            
        except ValueError:
            print("숫자를 정확히 입력해주세요.")
    
    return ingredients

# 재료 정보 입력
ingredients = input_ingredients()

if not ingredients:
    print("❌ 재료 정보가 입력되지 않았습니다. 샘플 데이터를 생성합니다.")
    ingredients = [
        {'name': '김치', 'current_stock': 50, 'daily_usage': 5, 'lead_time_days': 3},
        {'name': '돼지고기', 'current_stock': 30, 'daily_usage': 8, 'lead_time_days': 2},
        {'name': '두부', 'current_stock': 20, 'daily_usage': 4, 'lead_time_days': 1},
        {'name': '파', 'current_stock': 15, 'daily_usage': 2, 'lead_time_days': 2},
        {'name': '마늘', 'current_stock': 10, 'daily_usage': 1, 'lead_time_days': 3}
    ]
    print("✅ 샘플 재료 데이터 생성 완료")

# 데이터프레임 생성
df = pd.DataFrame(ingredients)
print(f"\n📊 입력된 재료 정보 ({len(df)}개)")
display(df)

## Step 2: 7일치 예측 재고 계산

In [ ]:
# 재고 예측 계산 함수
def calculate_inventory_forecast(df, days=7):
    forecast_data = []
    
    for _, ingredient in df.iterrows():
        name = ingredient['name']
        current_stock = ingredient['current_stock']
        daily_usage = ingredient['daily_usage']
        
        # 7일간 재고 예측
        for day in range(days):
            date = datetime.now() + timedelta(days=day)
            predicted_stock = max(0, current_stock - (daily_usage * day))
            
            forecast_data.append({
                'date': date.strftime('%m-%d'),
                'ingredient': name,
                'day': day + 1,
                'predicted_stock': predicted_stock
            })
    
    return pd.DataFrame(forecast_data)

# 7일간 재고 예측 계산
forecast_df = calculate_inventory_forecast(df, 7)

print("📈 7일간 재고 예측")
# 피벗 테이블로 보기 좋게 정리
forecast_pivot = forecast_df.pivot(index='ingredient', columns='day', values='predicted_stock').round(2)
forecast_pivot.columns = [f'{i}일차' for i in range(1, 8)]
display(forecast_pivot)

## Step 3: 발주 필요일 및 발주량 자동 출력

In [ ]:
# 발주 계산 함수
def calculate_reorder_info(df):
    reorder_info = []
    
    for _, ingredient in df.iterrows():
        name = ingredient['name']
        current_stock = ingredient['current_stock']
        daily_usage = ingredient['daily_usage']
        lead_time = ingredient['lead_time_days']
        
        # 안전재고 = 일일 사용량 × 리드타임 × 1.2
        safety_stock = daily_usage * lead_time * 1.2
        
        # 재주문점 = 안전재고
        reorder_point = safety_stock
        
        # 발주 필요일 계산
        if current_stock <= reorder_point:
            days_until_reorder = 0  # 이미 발주 필요
            reorder_needed = True
        else:
            days_until_reorder = (current_stock - reorder_point) / daily_usage
            reorder_needed = days_until_reorder <= 0
        
        # 발주량 계안 (안전재고 + 리드타임 동안 사용량)
        reorder_quantity = safety_stock + (daily_usage * lead_time)
        
        reorder_info.append({
            'name': name,
            'current_stock': current_stock,
            'daily_usage': daily_usage,
            'lead_time': lead_time,
            'safety_stock': safety_stock,
            'reorder_point': reorder_point,
            'days_until_reorder': max(0, days_until_reorder),
            'reorder_quantity': reorder_quantity,
            'reorder_needed': reorder_needed
        })
    
    return pd.DataFrame(reorder_info)

# 발주 정보 계산
reorder_df = calculate_reorder_info(df)

# 컬럼명 한글화
reorder_display = reorder_df.copy()
reorder_display.columns = ['재료명', '현재재고', '일일사용량', '소요일수', '안전재고', '재주문점', '발주까지일수', '발주량', '발주필요']

# 소수점 2자리까지 표시
for col in ['현재재고', '일일사용량', '안전재고', '재주문점', '발주까지일수', '발주량']:
    reorder_display[col] = reorder_display[col].round(2)

print("🚨 발주 필요 정보")
display(reorder_display)

# 긴급 발주 필요 재료 표시
urgent_reorder = reorder_display[reorder_display['발주필요'] == True]

if not urgent_reorder.empty:
    print("\n⚠️ 긴급 발주 필요 재료:")
    for _, item in urgent_reorder.iterrows():
        print(f"  • {item['재료명']}: 현재 {item['현재재고']}, {item['발주량']:.0f} 발주 권장")
else:
    print("\n✅ 현재 긴급 발주 필요 재료 없음")

## Step 4: 재고 추이 및 발주점 시각화

In [ ]:
# 시각화 함수
def plot_inventory_forecast(forecast_df, reorder_df):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. 전체 재료별 재고 추이
    ax1 = axes[0, 0]
    for ingredient in forecast_df['ingredient'].unique():
        data = forecast_df[forecast_df['ingredient'] == ingredient]
        ax1.plot(data['day'], data['predicted_stock'], 
                marker='o', linewidth=2, markersize=4, label=ingredient)
    
    ax1.set_title('7일간 재고 예측 추이', fontsize=14, fontweight='bold')
    ax1.set_xlabel('예측 일수')
    ax1.set_ylabel('예상 재고량')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # 2. 발주 필요 여부 파이차트
    ax2 = axes[0, 1]
    reorder_counts = reorder_df['reorder_needed'].value_counts()
    labels = ['발주 필요', '재고 충분']
    colors = ['#ff6b6b', '#51cf66']
    
    ax2.pie([reorder_counts.get(True, 0), reorder_counts.get(False, 0)], 
            labels=[labels[i] for i in range(2) if i < len(reorder_counts)], 
            colors=colors[:len(reorder_counts)], autopct='%1.1f%%', startangle=90)
    ax2.set_title('발주 필요 재료 비율', fontsize=14, fontweight='bold')
    
    # 3. 현재재고 vs 재주문점 비교
    ax3 = axes[1, 0]
    x_pos = np.arange(len(reorder_df))
    width = 0.35
    
    bars1 = ax3.bar(x_pos - width/2, reorder_df['current_stock'], width, 
                    label='현재 재고', color='#4dabf7')
    bars2 = ax3.bar(x_pos + width/2, reorder_df['reorder_point'], width,
                    label='재주문점', color='#ff6b6b', alpha=0.7)
    
    ax3.set_title('현재 재고 vs 재주문점', fontsize=14, fontweight='bold')
    ax3.set_xlabel('재료')
    ax3.set_ylabel('수량')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(reorder_df['name'], rotation=45)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 막대 위에 숫자 표시
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{height:.0f}', ha='center', va='bottom', fontsize=9)
    
    # 4. 일일사용량 순위
    ax4 = axes[1, 1]
    sorted_df = reorder_df.sort_values('daily_usage', ascending=True)
    bars = ax4.barh(sorted_df['name'], sorted_df['daily_usage'], color='#74c0fc')
    
    ax4.set_title('일일 사용량 순위', fontsize=14, fontweight='bold')
    ax4.set_xlabel('일일 사용량')
    ax4.set_ylabel('재료')
    
    # 막대 옆에 숫자 표시
    for bar, value in zip(bars, sorted_df['daily_usage']):
        ax4.text(value + 0.1, bar.get_y() + bar.get_height()/2,
                f'{value:.0f}', ha='left', va='center', fontsize=9)
    
    plt.tight_layout()
    return fig

# 차트 생성
fig = plot_inventory_forecast(forecast_df, reorder_df)
plt.show()

# 최종 요약 통계
total_current_stock = reorder_df['current_stock'].sum()
total_daily_usage = reorder_df['daily_usage'].sum()
days_of_supply = total_current_stock / total_daily_usage if total_daily_usage > 0 else 0
urgent_items = reorder_df[reorder_df['reorder_needed'] == True]['name'].tolist()

print(f"\n📊 재고 관리 요약")
print(f"총 현재 재고: {total_current_stock:.0f}")
print(f"총 일일 사용량: {total_daily_usage:.0f}")
print(f"총 공급 가능 일수: {days_of_supply:.1f}일")
print(f"긴급 발주 필요 재료: {len(urgent_items)}개 ({', '.join(urgent_items) if urgent_items else '없음'})")

## Step 5: CSV 다운로드

In [ ]:
# 데이터 저장 및 다운로드
from datetime import datetime
import os

# 파일 이름 생성
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_filename = f"재고예측_발주계산_{timestamp}.csv"
chart_filename = f"재고차트_{timestamp}.png"

# CSV 데이터 준비
export_data = []

# 현재 재고 정보 추가
for _, item in reorder_df.iterrows():
    export_data.append({
        '구분': '현재재고정보',
        '재료명': item['name'],
        '현재재고': item['current_stock'],
        '일일사용량': item['daily_usage'],
        '소요일수': item['lead_time'],
        '안전재고': item['safety_stock'],
        '재주문점': item['reorder_point'],
        '발주까지일수': item['days_until_reorder'],
        '발주량': item['reorder_quantity'],
        '발주필요': '예' if item['reorder_needed'] else '아니오'
    })

# 7일 예측 정보 추가
for day in range(1, 8):
    day_data = forecast_df[forecast_df['day'] == day]
    for _, item in day_data.iterrows():
        export_data.append({
            '구분': f'{day}일차예측',
            '재료명': item['ingredient'],
            '예상재고': item['predicted_stock'],
            '일일사용량': '',
            '소요일수': '',
            '안전재고': '',
            '재주문점': '',
            '발주까지일수': '',
            '발주량': '',
            '발주필요': ''
        })

# CSV 파일 저장
export_df = pd.DataFrame(export_data)
export_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
print(f"✅ CSV 파일 저장 완료: {csv_filename}")

# 차트 PNG 저장
fig.savefig(chart_filename, dpi=300, bbox_inches='tight')
print(f"✅ 차트 이미지 저장 완료: {chart_filename}")

In [ ]:
# 다운로드 기능 (Colab 환경에서만 작동)
try:
    from google.colab import files
    print("\n📥 파일 다운로드")
    files.download(csv_filename)
    files.download(chart_filename)
except ImportError:
    print(f"\n💾 파일이 로컬에 저장되었습니다:")
    print(f"- CSV: {csv_filename}")
    print(f"- 차트: {chart_filename}")
    
# 최종 확인
print(f"\n🎉 재고 예측 및 발주량 계산 완료!")
print(f"📊 처리된 재료: {len(df)}개")
print(f"📈 예측 기간: 7일")
print(f"🚨 긴급 발주 필요: {len(urgent_items)}개")

# 긴급 발구 알림
if urgent_items:
    print("\n⚠️ 긴급 발주 권장:")
    for _, item in urgent_reorder.iterrows():
        print(f"  • {item['재료명']}: {item['발주량']:.0f} 즉시 발주 권장")
else:
    print("\n✅ 현재 모든 재고가 안정적인 수준입니다.")